In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/ML-FinalAss/Walmart-Sales-Forecasting"

!pip install -U kaggle mlflow dagshub lightgbm

Mounted at /content/drive
/content/drive/MyDrive/ML-FinalAss/Walmart-Sales-Forecasting
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import zipfile
import glob

data_dir = "data"
required_files = ["train.csv", "test.csv", "stores.csv", "features.csv"]

files_exist = os.path.exists(data_dir) and all(
    os.path.exists(os.path.join(data_dir, f)) for f in required_files
)

if files_exist:
    print("Dataset is already downloaded and extracted in the 'data' folder. Skipping download!")
else:
    print("Dataset not found. Starting download...")

    os.environ['KAGGLE_API_TOKEN'] = "KGAT_caf00baa57d3f767dc15d31b24d4e730"

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting

    os.makedirs(data_dir, exist_ok=True)
    main_zip = "walmart-recruiting-store-sales-forecasting.zip"
    if os.path.exists(main_zip):
        with zipfile.ZipFile(main_zip, "r") as zip_ref:
            zip_ref.extractall(data_dir)
        os.remove(main_zip)
    else:
        print("The zip file wasn't found. Double check your Kaggle Token expiration status!")

    inner_zips = glob.glob(os.path.join(data_dir, "*.zip"))
    for file in inner_zips:
        with zipfile.ZipFile(file, "r") as zip_ref:
            zip_ref.extractall(data_dir)
        os.remove(file)

    print("Success! All datasets downloaded and extracted:")
    print(os.listdir(data_dir))

Dataset is already downloaded and extracted in the 'data' folder. Skipping download!


In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import mlflow
import dagshub
import mlflow.sklearn

dagshub.init(repo_owner='slosa23', repo_name='Walmart-Sales-Forecasting', mlflow=True)

train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")

mlflow.set_experiment("LightGBM_Training_V1")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=4b2714df-6562-4dff-8335-7236b6df02f2&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=fe3a2c5532bd1e58946ce932dfffa0f55b361fa96d51b478374f981ec2ecf340




Accessing as slosa23

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

2026/07/07 14:24:30 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM_Training_V1' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/301c55eeeabe427b9138642fd5a67462', creation_time=1783434270868, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1783434270868, lifecycle_stage='active', name='LightGBM_Training_V1', tags={}, trace_location=None, workspace='default'>

In [8]:
import yaml
import mlflow
import mlflow.sklearn
from src.features.preprocessing import get_model_ready_data
from src.models.lightgbm_pipeline import create_lightgbm_pipeline
from src.evaluation.metrics import calculate_wmae

with open("configs/lightgbm_config.yaml", "r") as f:
    config = yaml.safe_load(f)

model_name = config['model']['name']
model_params = config['model']['params']
split_date = config['data']['split_date']

with mlflow.start_run(run_name=model_name):

    X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
        train_raw, stores, features, split_date=split_date
    )

    pipeline = create_lightgbm_pipeline(preprocessor, model_params)

    mlflow.log_params(model_params)
    print(f"Training production pipeline: {model_name}...")
    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_val)
    production_wmae = calculate_wmae(y_val, preds, is_holiday_val)
    print(f"LightGBM Production Validation WMAE: {production_wmae:.2f}")

    mlflow.log_metric("WMAE", production_wmae)

    trusted_types = [
        'collections.OrderedDict',
        'lightgbm.basic.Booster',
        'lightgbm.sklearn.LGBMRegressor',
        'sklearn.compose._column_transformer._RemainderColsList'
    ]

    mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="model",
        skops_trusted_types=trusted_types,
        registered_model_name=model_name
    )

    print(f"'{model_name}' has been registered to the DagsHub")

Training production pipeline: Walmart_LightGBM_Baseline...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025667 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2450
[LightGBM] [Info] Number of data points in the train set: 294132, number of used features: 23
[LightGBM] [Info] Start training from score 16105.306894


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LightGBM Production Validation WMAE: 14167.88


Successfully registered model 'Walmart_LightGBM_Baseline'.
2026/07/07 14:30:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_LightGBM_Baseline, version 1
Created version '1' of model 'Walmart_LightGBM_Baseline'.


'Walmart_LightGBM_Baseline' has been registered to the DagsHub
🏃 View run Walmart_LightGBM_Baseline at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2/runs/da715d6d2b074537b072e4359b85728e
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2


In [9]:
import yaml
import mlflow
import mlflow.sklearn
from src.features.preprocessing import get_model_ready_data
from src.models.lightgbm_pipeline import create_lightgbm_pipeline
from src.evaluation.metrics import calculate_wmae

with open("configs/lightgbm_config.yaml", "r") as f:
    config = yaml.safe_load(f)

model_name = config['model']['name']
base_params = config['model']['params']
split_date = config['data']['split_date']

X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
    train_raw, stores, features, split_date=split_date
)

learning_rates = [0.05, 0.1]
num_leaves_options = [31, 127]

best_wmae = float('inf')
best_pipeline = None
best_params = None

for lr in learning_rates:
    for leaves in num_leaves_options:

        current_params = base_params.copy()
        current_params['learning_rate'] = lr
        current_params['num_leaves'] = leaves

        run_title = f"{model_name}_LR_{lr}_Leaves_{leaves}"

        with mlflow.start_run(run_name=run_title):
            pipeline = create_lightgbm_pipeline(preprocessor, current_params)
            mlflow.log_params(current_params)

            pipeline.fit(X_train, y_train)

            preds = pipeline.predict(X_val)
            current_wmae = calculate_wmae(y_val, preds, is_holiday_val)
            mlflow.log_metric("WMAE", current_wmae)

            print(f"{run_title} completed. WMAE: {current_wmae:.2f}")

            if current_wmae < best_wmae:
                best_wmae = current_wmae
                best_pipeline = pipeline
                best_params = current_params

print(f"\n Sweep finished, Champion WMAE: {best_wmae:.2f}")

with mlflow.start_run(run_name=model_name):
    mlflow.log_params(best_params)
    mlflow.log_metric("WMAE", best_wmae)

    trusted_types = [
        'collections.OrderedDict',
        'lightgbm.basic.Booster',
        'lightgbm.sklearn.LGBMRegressor',
        'sklearn.compose._column_transformer._RemainderColsList'
    ]

    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        name="model",
        skops_trusted_types=trusted_types,
        registered_model_name=model_name
    )

    print(f"model successfully published to the DagsHub")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.088483 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2450
[LightGBM] [Info] Number of data points in the train set: 294132, number of used features: 23
[LightGBM] [Info] Start training from score 16105.306894


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Walmart_LightGBM_Baseline_LR_0.05_Leaves_31 completed. WMAE: 14110.87
🏃 View run Walmart_LightGBM_Baseline_LR_0.05_Leaves_31 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2/runs/852385965fdc46559f620732ea8e2857
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.036754 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2450
[LightGBM] [Info] Number of data points in the train set: 294132, number of used features: 23
[LightGBM] [Info] Start training from score 16105.306894


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Walmart_LightGBM_Baseline_LR_0.05_Leaves_127 completed. WMAE: 14199.10
🏃 View run Walmart_LightGBM_Baseline_LR_0.05_Leaves_127 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2/runs/14bdd236cb654e1fbd15aaf244e92162
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025752 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2450
[LightGBM] [Info] Number of data points in the train set: 294132, number of used features: 23
[LightGBM] [Info] Start training from score 16105.306894


...

Walmart_LightGBM_Baseline_LR_0.1_Leaves_31 completed. WMAE: 14083.46
🏃 View run Walmart_LightGBM_Baseline_LR_0.1_Leaves_31 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2/runs/0e63f10d2f864aa3a3647e29fea2b74e
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025782 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2450
[LightGBM] [Info] Number of data points in the train set: 294132, number of used features: 23
[LightGBM] [Info] Start training from score 16105.306894


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Walmart_LightGBM_Baseline_LR_0.1_Leaves_127 completed. WMAE: 14179.99
🏃 View run Walmart_LightGBM_Baseline_LR_0.1_Leaves_127 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2/runs/7c0ecd04f7d24a1896eebd4ba4bdac1d
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2

 Sweep finished, Champion WMAE: 14083.46


Registered model 'Walmart_LightGBM_Baseline' already exists. Creating a new version of this model...
2026/07/07 14:38:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_LightGBM_Baseline, version 2
Created version '2' of model 'Walmart_LightGBM_Baseline'.


model successfully published to the DagsHub
🏃 View run Walmart_LightGBM_Baseline at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2/runs/b5de6e25450d46eeb73cde15d3de05b2
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2


In [15]:
import yaml
import mlflow
import mlflow.sklearn
from src.features.preprocessing import get_model_ready_data
from src.models.lightgbm_pipeline import create_lightgbm_pipeline
from src.evaluation.metrics import calculate_wmae

with open("configs/lightgbm_config.yaml", "r") as f:
    config = yaml.safe_load(f)

model_name = config['model']['name']
model_params = config['model']['params']
split_date = config['data']['split_date']

with mlflow.start_run(run_name=model_name):
    X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
        train_raw, stores, features, split_date=split_date
    )

    pipeline = create_lightgbm_pipeline(preprocessor, model_params)
    mlflow.log_params(model_params)

    print(f"Training experimental run: {model_name}...")
    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_val)
    production_wmae = calculate_wmae(y_val, preds, is_holiday_val)
    print(f"\n Experimental Run Validation WMAE: {production_wmae:.2f}")

    mlflow.log_metric("WMAE", production_wmae)

    trusted_types = [
        'collections.OrderedDict',
        'lightgbm.basic.Booster',
        'lightgbm.sklearn.LGBMRegressor',
        'sklearn.compose._column_transformer._RemainderColsList'
    ]

    mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="model",
        skops_trusted_types=trusted_types,
        registered_model_name="Walmart_LightGBM_Baseline"
    )

    print(f"Run saved to the DagsHub Model Registry")

Training experimental run: Walmart_LightGBM_Iteration_4...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024122 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2450
[LightGBM] [Info] Number of data points in the train set: 294132, number of used features: 23
[LightGBM] [Info] Start training from score 16105.306894
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Ligh

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



 Experimental Run Validation WMAE: 14162.16


Registered model 'Walmart_LightGBM_Baseline' already exists. Creating a new version of this model...
2026/07/07 14:56:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_LightGBM_Baseline, version 3
Created version '3' of model 'Walmart_LightGBM_Baseline'.


Run saved to the DagsHub Model Registry
🏃 View run Walmart_LightGBM_Iteration_4 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2/runs/3f9e311a88c04a598165dc7b57628781
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/2
